# Case Study — DITU Funnel & Cross-Media Audience Hub
**Company (context):** Caracol TV · **Synthetic / random audience consumption data**

This notebook walks through a **media analytics workflow**: create event-level audience data, explore consumption, build a Cross-Media KPI panel, model a DITU funnel, and score audiences for targeting.

**Why this notebook matters**  
In media companies, decisions about programming, paid promotion, and streaming growth only work when teams share the same definitions of *reach*, *engagement*, and *valuable users*. This case study shows how event-level consumption becomes executive KPIs.

> All figures are **fictional** and seeded for reproducibility (`SEED = 42`).


## 0. Setup

**Why this step is important**  
Before any analysis, we fix the environment so the work is **reproducible and portable**:

- A fixed `SEED` makes random data regenerable (same results for demos, interviews, or QA).
- Explicit `Path` handling avoids “file not found” when the notebook runs from `notebooks/` or the portfolio root.
- Creating `DATA_DIR` up front guarantees exports land in a known place for BI tools or other notebooks.

Without this step, colleagues cannot rerun your analysis reliably.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

# Works from notebooks/ or portfolio root
ROOT = Path.cwd()
if (ROOT / "data").exists():
    DATA_DIR = ROOT / "data"
elif (ROOT.parent / "data").exists():
    DATA_DIR = ROOT.parent / "data"
else:
    DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = DATA_DIR / "audience_consumption_random.csv"
print("Data folder:", DATA_DIR.resolve())
print("Output CSV:", OUT_CSV.name)


## 1. Generate random audience consumption

**Why this step is important**  
Real Cross-Media work starts from **event-level logs** (who watched what, where, and for how long)—not from pre-aggregated dashboards.

We simulate that grain so we can practice:

- Designing realistic dimensions (channel, title, genre, device, demographics).
- Modeling different session lengths by channel (TV/Radio vs YouTube clips).
- Exporting a reusable CSV as a “bronze” dataset for later steps.

**Business value:** if you only ever see weekly totals, you cannot diagnose *why* reach moved. Event data lets you answer that.

Each row = one **consumption event** (a user watching a title on a channel).

In [ ]:
N_EVENTS = 25_000
N_USERS = 5_000

channels = ["TV", "DITU", "YouTube", "Portal", "Radio"]
channel_p = [0.28, 0.22, 0.25, 0.12, 0.13]

genres = ["Novela", "News", "Sports", "Entertainment", "Kids", "Documentary"]
devices = ["Smart TV", "Mobile", "Desktop", "Tablet", "Radio set"]
titles = [
    "Morning News Live", "Prime Novela A", "Prime Novela B", "Match Night",
    "Talk Show Central", "Kids Club", "Weekend Special", "Late Night Desk",
    "Regional Report", "Streaming Exclusive 01", "Streaming Exclusive 02",
    "YouTube Clip Hour", "Portal Catch-up", "Radio Morning Drive",
]

ages = rng.integers(16, 75, N_USERS)
cities = rng.choice(
    ["Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Other"],
    N_USERS,
    p=[0.34, 0.18, 0.14, 0.10, 0.08, 0.16],
)
genders = rng.choice(["F", "M", "Other"], N_USERS, p=[0.51, 0.47, 0.02])

user_lookup = pd.DataFrame({
    "user_id": [f"U{i:05d}" for i in range(1, N_USERS + 1)],
    "age": ages,
    "gender": genders,
    "city": cities,
})

# Event timestamps over ~90 days
start = np.datetime64("2026-01-01")
offsets_min = rng.integers(0, 90 * 24 * 60, N_EVENTS)
event_ts = start + offsets_min.astype("timedelta64[m]")

chosen_users = rng.integers(0, N_USERS, N_EVENTS)
chosen_channels = rng.choice(channels, N_EVENTS, p=channel_p)

# Watch minutes depend on channel (TV/Radio longer sessions on average)
base_watch = {
    "TV": (25, 12),
    "DITU": (38, 18),
    "YouTube": (12, 8),
    "Portal": (18, 10),
    "Radio": (40, 20),
}
watch_minutes = np.array([
    max(1, int(rng.normal(base_watch[ch][0], base_watch[ch][1])))
    for ch in chosen_channels
])

consumption = pd.DataFrame({
    "event_id": [f"E{i:06d}" for i in range(1, N_EVENTS + 1)],
    "event_ts": event_ts,
    "user_id": user_lookup.loc[chosen_users, "user_id"].to_numpy(),
    "channel": chosen_channels,
    "title": rng.choice(titles, N_EVENTS),
    "genre": rng.choice(genres, N_EVENTS),
    "device": rng.choice(devices, N_EVENTS, p=[0.33, 0.38, 0.14, 0.08, 0.07]),
    "watch_minutes": watch_minutes,
    "completed": rng.random(N_EVENTS) < 0.37,
})

# Attach demographics
consumption = consumption.merge(user_lookup, on="user_id", how="left")
consumption["date"] = pd.to_datetime(consumption["event_ts"]).dt.floor("D")
consumption["hour"] = pd.to_datetime(consumption["event_ts"]).dt.hour

consumption.to_csv(OUT_CSV, index=False)
print(f"Saved {len(consumption):,} random consumption events → {OUT_CSV}")
consumption.head(10)


## 2. Explore audience consumption

**Why this step is important**  
Exploration is quality control + product sense:

- **Shape / channel mix** shows whether volume and distribution look plausible.
- **Watch-time distribution** reveals outliers or unrealistic session lengths.
- **Top titles** surface what content actually drives attention.

Skipping EDA is how teams ship dashboards that look polished but hide broken joins or impossible metrics.

In [ ]:
print("Shape:", consumption.shape)
print("\nChannels:")
print(consumption["channel"].value_counts())
print("\nWatch minutes describe:")
display(consumption["watch_minutes"].describe().round(1))
print("\nTop titles by total watch hours:")
(
    consumption.groupby("title")["watch_minutes"].sum()
    .div(60).sort_values(ascending=False).head(8).round(1)
)


## 2.1 Build the daily Cross-Media panel

**Why this step is important**  
Executives do not read 25,000 event rows — they need a **shared KPI layer** by day and channel:

- `reach` = unique users (not just clicks/events).
- `watch_hours` = intensity of consumption.
- `completion_rate` = quality of engagement.
- `reach_share` = how attention is split across TV, DITU, YouTube, Portal, Radio.

This panel enables **Cross-Media decisions**: where to promote a title, which platform is growing, and whether digital complements linear.

In [ ]:
# Daily Cross-Media panel from event-level consumption
cross = (
    consumption.groupby(["date", "channel"], as_index=False)
    .agg(
        reach=("user_id", "nunique"),
        events=("event_id", "count"),
        watch_hours=("watch_minutes", lambda s: round(s.sum() / 60, 2)),
        completion_rate=("completed", "mean"),
    )
)
cross["completion_rate"] = cross["completion_rate"].round(3)
display(cross.head())

channel_share = (
    cross.groupby("channel")["reach"].sum().sort_values(ascending=False)
)
(channel_share / channel_share.sum()).rename("reach_share").round(3)


## 3. DITU digital funnel

**Why this step is important**  
Streaming growth is a **conversion problem**, not only a reach problem. The funnel shows where users drop:

Visit → Signup → Activate → Watch (7d) → Return (30d)

Each stage answers a different business question:

- Poor signup → friction in registration / paywall.
- Poor activation → onboarding / content discovery.
- Poor return → retention / catalog / habit formation.

**Why it matters for Caracol/DITU-style products:** marketing can buy visits, but product & content own activation and retention. The funnel assigns ownership.

In [ ]:
ditu_users = consumption.loc[consumption["channel"] == "DITU", "user_id"].unique()
n_visit = max(len(ditu_users), 1)

# Funnel rates inspired by streaming products (illustrative)
stages = ["Visit", "Signup", "Activate", "Watch_7d", "Return_30d"]
rates = [1.00, 0.38, 0.62, 0.55, 0.41]
counts, remaining = [], n_visit
for r in rates:
    remaining = int(remaining * r)
    counts.append(remaining)

funnel = pd.DataFrame({"stage": stages, "users": counts})
funnel["conv_from_prev"] = (funnel["users"] / funnel["users"].shift(1)).round(3)
funnel


## 4. Audience score from consumption behavior

**Why this step is important**  
Not every user is equally valuable for campaigns or personalization. A transparent score combines:

- Frequency (`sessions`)
- Depth (`watch_hours`)
- Breadth (`channels`)
- Quality (`completion_rate`)

Then we bucket users into **Low / Mid / High**.

**Business value:** High-value segments can be prioritized for premium campaigns, exclusive content tests, or retention plays — instead of treating the whole audience as average.

In [ ]:
user_feat = (
    consumption.groupby("user_id", as_index=False)
    .agg(
        sessions=("event_id", "count"),
        watch_hours=("watch_minutes", lambda s: s.sum() / 60),
        channels=("channel", "nunique"),
        completion_rate=("completed", "mean"),
        age=("age", "first"),
        city=("city", "first"),
    )
)

user_feat["score"] = (
    0.30 * (user_feat["sessions"] / user_feat["sessions"].max())
    + 0.40 * (user_feat["watch_hours"] / user_feat["watch_hours"].max())
    + 0.15 * (user_feat["channels"] / user_feat["channels"].max())
    + 0.15 * user_feat["completion_rate"].fillna(0)
) * 100

user_feat["segment"] = pd.cut(
    user_feat["score"],
    bins=[-0.1, 40, 70, 100],
    labels=["Low", "Mid", "High"],
)

seg_share = user_feat["segment"].value_counts(normalize=True).sort_index().round(3)
display(seg_share)
user_feat.sort_values("score", ascending=False).head(8)


## 5. Executive summary

**Why this step is important**  
The last mile of analytics is **communication**. A compact KPI dictionary:

- Fits a slide or Looker tile.
- Aligns data, content, and business teams on the same numbers.
- Creates a checklist for monitoring (reach, signup rate, high-value share, top channel).

If stakeholders cannot remember the story in five metrics, the analysis will not drive decisions — no matter how sophisticated the code was.

In [ ]:
daily_reach = cross.groupby("date")["reach"].sum()
summary = {
    "events": int(len(consumption)),
    "unique_users": int(consumption["user_id"].nunique()),
    "avg_daily_reach": int(daily_reach.mean()),
    "avg_watch_min_per_event": round(float(consumption["watch_minutes"].mean()), 1),
    "ditu_signup_rate": float(funnel.loc[1, "conv_from_prev"]),
    "high_value_audience_share": float(seg_share.get("High", 0)),
    "top_channel_by_reach": channel_share.idxmax(),
    "csv_path": str(OUT_CSV.resolve()),
}
summary
